In [1]:
from platform import python_version
print(python_version())

3.11.14


### Calculating DEGs statistics

### For each LFC/FDR cutoff set, we get a different set of DEGs
  - LFC: LFC cutoff and FDR_LFC cutoff
  - Pathway: fdr and pval pathway cutoff and min num of genes

### Up and Down DEGs simulation
  - Up and Down DEGs/DAPs
  - Up and Down in pathways

### there are 2 statistical tables
  - pval/fdr cutoff x degs
  - pval/fdr/geneset/quantile degs_in_pathway, num_pathways

In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.GDC_lib import GDC
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']


case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor
>>> case Tumor
	DEGs 19572
		Up (#9553)
		Dw (#10019)

Up-regulated per biotype
                               biotype     n
0                            IG_C_gene    13
1                      IG_C_pseudogene     2
2                            IG_J_gene    11
3                      IG_J_pseudogene     1
4                            IG_V_gene   122
5                      IG_V_pseudogene    42
6                              Mt_tRNA    13
7                                  TEC    96
8                            TR_C_gene     6
9                            TR_D_gene     1
10                           TR_J_gene     9
11                           TR_V_gene    43
12                     TR_V_pseudogene     2
13                              lncRNA  3139
14                               miRNA    77
15                            misc_RNA    15
16              polymorphic_pseudogene     4
17        

In [5]:
gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA)

gdc.memory_restriction = False

### Get all programs

In [6]:
force = False
verbose = False

prog_list = gdc.get_gdc_progams(force=force, verbose=verbose)

df_psi = gdc.get_primary_sites(prog_id=PROG_ID, force=False, verbose=verbose)
print(len(df_psi))

df_psi.shape, df_psi.columns

33


((33, 5),
 Index(['psi_id', 'primary_site', 'project_id', 'disease_type', 'name'], dtype='object'))

In [7]:
gdc.set_primary_site(psi_id=PSI_ID, verbose=verbose)

True

In [ ]:
verbose=False

imax_samples = 32

# df_tumor, df_normal, df_gtex_ctrl = gdc.get_file_expression_both_tumor_and_normal(imax_samples=imax_samples, verbose=verbose)

In [18]:
dic_tumor, dic_normal = gdc.get_dic_expression_tumor_and_normal(verbose=verbose)
len(dic_tumor)

Dowloading normal files: 0.
Dowloading tumor files: 0..........10..........20..........30..


32

In [ ]:
df_tumor, df_normal = gdc.prepare_normal_tumor_tables(
            dic_tumor, dic_normal, imax_tumor=imax_samples, imax_normal=imax_samples, verbose=verbose
        )

In [35]:
from collections import Counter
import pandas as pd
import math

def get_representative_geneids(
    dfs: list[pd.DataFrame],
    min_fraction: float = 0.75,
) -> pd.DataFrame:
    """
    Return genes present in more than min_fraction of dataframes.

    For 10 dataframes and min_fraction=0.75:
    strict >75% means present in at least 8 dataframes.

    Presence is counted once per dataframe, even if duplicated inside a dataframe.
    """

    gene_cols: list[str] = ["geneid", "symbol"]

    n = len(dfs)
    if n == 0:
        return pd.DataFrame(columns=gene_cols + ["n_dfs", "fraction"])

    min_count = math.floor(n * min_fraction) + 1  # strict > min_fraction

    counter = Counter()

    for df in dfs:
        missing = [c for c in gene_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Missing columns in dataframe: {missing}")

        genes_in_df = (
            df[gene_cols]
            .dropna(subset=gene_cols)
            .astype(str)
            .drop_duplicates()
        )

        counter.update(map(tuple, genes_in_df.to_numpy()))

    result = (
        pd.DataFrame(
            [(geneid, symbol, count) for (geneid, symbol), count in counter.items()],
            columns=gene_cols + ["n_dfs"],
        )
        .assign(fraction=lambda x: x["n_dfs"] / n)
        .query("n_dfs >= @min_count")
        .sort_values(["n_dfs"] + gene_cols, ascending=[False] + [True] * len(gene_cols))
        .reset_index(drop=True)
    )

    return result

In [27]:
df_list = []

cols = ["geneid", "symbol", "biotype", "counts"]


i = 0
for _, dfa in dic_tumor.items():
    if dfa is None or dfa.empty:
        continue

    i += 1
    # print(i, end=' ')
    if "gene_id" in dfa.columns:
        dfa = dfa.rename(columns={"gene_id": "geneid"})
    if "gene_type" in dfa.columns:
        dfa = dfa.rename(columns={"gene_type": "biotype"})

    dfa = dfa[cols]
    df_list.append(dfa)

In [36]:
dfq = get_representative_geneids(df_list, min_fraction=0.75)

print(len(dfq))

dfq.head(3)

43341


,geneid,symbol,n_dfs,fraction
0,ENSG00000000003,TSPAN6,32,1.0
1,ENSG00000000005,TNMD,32,1.0
2,ENSG00000000419,DPM1,32,1.0


In [38]:
lista = np.unique(dfq.geneid.to_list())
print(len(lista))
lista[:3]

43341


array(['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419'],
      dtype='<U15')

In [ ]:
df_tumor = pd.DataFrame()

cols = ["geneid", "symbol", "counts"]
common_cols = ["geneid", "symbol"]

imax_tumor=50

i = 0
for _, dfa in dic_tumor.items():
    if dfa is None or dfa.empty:
        continue

    i += 1
    # print(i, end=' ')
    if "gene_id" in dfa.columns:
        dfa = dfa.rename(columns={"gene_id": "geneid"})
    if "gene_type" in dfa.columns:
        dfa = dfa.rename(columns={"gene_type": "biotype"})

    dfa = dfa[cols]

    dfa = (
        dfa.dropna(subset=['geneid', 'symbol'])
        .drop_duplicates(['geneid', 'symbol'])
    )

    dfa = dfa.rename(columns={"counts": f"tumor_{i}"})


    dfa = dfa[dfa.geneid.isin(lista)]
    dfa.reset_index(drop=True, inplace=True)

    if df_tumor.empty:
        df_tumor = dfa
    else:
        if i <= imax_tumor:
            df_tumor = df_tumor.merge(dfa, on=common_cols, how="outer")
            print(i, dfa.shape, df_tumor.shape)
        else:
            if verbose:
                print(">>> dfa", len(dfa), ",".join(dfa.symbol[:30]))
            break


2 (43341, 3) (43341, 4)
3 (43341, 3) (43341, 5)
4 (43341, 3) (43341, 6)
5 (43341, 3) (43341, 7)
6 (43341, 3) (43341, 8)
7 (10212, 3) (43341, 9)
8 (43341, 3) (43341, 10)
9 (43341, 3) (43341, 11)
10 (43341, 3) (43341, 12)
11 (43341, 3) (43341, 13)
12 (42767, 3) (43342, 14)


In [25]:
df_tumor.head(3)

,geneid,symbol,biotype,tumor_1,tumor_2,tumor_3,tumor_4,tumor_5,tumor_6,tumor_7,tumor_8,tumor_9,tumor_10,tumor_11,tumor_12
0,ENSG00000000003,TSPAN6,protein_coding,1644,622,839,618,643,728,800,856,909,2336,621,1671.0
1,ENSG00000000005,TNMD,protein_coding,5,1,0,64,4,0,0,1,1,1,0,86.0
2,ENSG00000000419,DPM1,protein_coding,985,782,956,674,297,633,1019,994,452,1172,489,1290.0


In [ ]:
dff_tumor = gdc.dff_tumor
len(dff_tumor)

In [ ]:
len(gdc.dic_tumor)

In [ ]:
print(len(df_tumor))
df_tumor.head(3)

In [ ]:
print(len(df_tumor.columns)-3)
df_tumor.head(3)

In [ ]:
print(len(df_normal.columns)-3)
df_normal.head(3)

In [ ]:
print(len(df_gtex_ctrl.columns)-2)
df_gtex_ctrl.head(3)

### CALC_DEGS -> build_counts_and_metadata()

In [ ]:
cdegs = CALC_DEGS(root_src=gdc.root_src, run_conda=False)
gdc.cdegs = cdegs

verbose=True
df_tumor, df_normal, msg = gdc.get_tumor_normal_tables(verbose=verbose)

In [ ]:
df_tumor.head(3)

In [ ]:
df_normal.head(3)

In [ ]:
df_counts, df_meta = cdegs.build_counts_and_metadata(
            df_tumor=df_tumor,
            df_normal=df_normal,
            how="inner"
        )

df_counts.head(3)

In [ ]:
df_meta

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind

def build_df_exp_and_filter(
    df_counts: pd.DataFrame,
    df_meta: pd.DataFrame,
    gene_col: str = "geneid",
    condition_col: str = "condition",
    sample_col: str = "sample",
    tumor_label: str = "tumor",
    normal_label: str = "normal",
    equal_var: bool = False,   # False = Welch t-test, safer when n differs
) -> tuple[pd.DataFrame, list, list]:
    df = df_counts.copy()

    # Samples by condition
    normal_samples = df_meta.loc[
        df_meta[condition_col] == normal_label, sample_col
    ].tolist()

    tumor_samples = df_meta.loc[
        df_meta[condition_col] == tumor_label, sample_col
    ].tolist()

    # Keep only samples present in df_counts
    normal_samples = [s for s in normal_samples if s in df.columns]
    tumor_samples = [s for s in tumor_samples if s in df.columns]

    sample_cols = normal_samples + tumor_samples

    ncols_normal = len(normal_samples)
    ncols_tumor  = len(tumor_samples)

    nmin_cols = min(ncols_normal, ncols_tumor)

    df["total"] = df[sample_cols].sum(axis=1)

    df = df.loc[
        df["total"] > nmin_cols * 25
    ].reset_index(drop=True, inplace=False)



    # Convert counts to numeric
    df[normal_samples + tumor_samples] = df[normal_samples + tumor_samples].apply(
        pd.to_numeric, errors="coerce"
    )

    # Optional but recommended for RNA-seq counts:
    # log-transform before t-test
    normal_mat = np.log2(df[normal_samples] + 1)
    tumor_mat = np.log2(df[tumor_samples] + 1)

    # Row-wise t-test: tumor vs normal
    t_stat, pval = ttest_ind(
        tumor_mat,
        normal_mat,
        axis=1,
        equal_var=equal_var,
        nan_policy="omit",
    )

    df["t_stat"] = t_stat
    df["pval"] = pval

    # Useful summaries
    df["mean_normal"] = normal_mat.mean(axis=1)
    df["mean_tumor"] = tumor_mat.mean(axis=1)
    df["lfc"] = df["mean_tumor"] - df["mean_normal"]
    df["abs_lfc"] = df["lfc"].abs()

    # Order by p-value
    df = df.sort_values("pval", ascending=True)

    # Keep the 40% lowest p-values
    df = df[df.lfc < 0.01]
    df.reset_index(drop=True, inplace=True)

    return df, normal_samples, tumor_samples


In [ ]:
dff, normal_samples, tumor_samples = build_df_exp_and_filter(df_counts=df_counts, df_meta=df_meta,
    gene_col = "geneid",
    equal_var = False,)   # False = Welch t-test, safer when n differs

print(len(df_counts), len(dff))

dff.head(3)

In [ ]:
import seaborn as sns

In [ ]:
cols = ['geneid'] + normal_samples + tumor_samples

dff2 = dff[cols].copy()
dff2.set_index('geneid', inplace=True)
dff2.head(3)

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import zscore

figsize = (12,8)

title = 'Hierarchical Clustering of Expression Data'

# numeric matrix
dff2 = dff2.apply(pd.to_numeric, errors="coerce").fillna(0)
mat = np.log2(dff2 + 1)

# gene-wise z-score using pandas/numpy
row_mean = mat.mean(axis=1)
row_std = mat.std(axis=1)

mat_z = mat.sub(row_mean, axis=0).div(row_std.replace(0, np.nan), axis=0)
mat_z = mat_z.replace([np.inf, -np.inf], np.nan).fillna(0)

cg = sns.clustermap(
    mat_z,
    metric="correlation",
    method="average",
    figsize=figsize,
    cmap="viridis",
    cbar=True,
)

title = "Hierarchical Clustering of Expression Data"
cg.figure.suptitle(title, y=1.02)

plt.show()


In [ ]:
mat_z